# Cointegration Analysis: ETH/ETC Pair

This notebook tests whether ETH and ETC are cointegrated and suitable for statistical arbitrage.

## Goals
1. Load historical ETH and ETC price data
2. Test for cointegration using Engle-Granger test
3. Calculate the spread and hedge ratio
4. Analyze mean-reversion properties
5. Visualize spread behavior and trading opportunities
6. Determine if this pair is suitable for pairs trading

In [ ]:
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

import polars as pl
import numpy as np
from statistical_arbitrage.data.bitvavo_client import BitvavoDataCollector
from statistical_arbitrage.analysis.cointegration import PairAnalysis, create_summary_report
from statistical_arbitrage.visualization.spread_plots import (
    plot_price_comparison,
    plot_spread_analysis,
    plot_scatter_with_regression,
    plot_zscore_distribution,
)

## 1. Load Data

Load the ETH and ETC data that we collected in the previous notebook.

In [ ]:
# Find the most recent data files
data_dir = project_root / "data" / "raw"
eth_files = sorted(data_dir.glob("ETH-EUR_*.parquet"))
etc_files = sorted(data_dir.glob("ETC-EUR_*.parquet"))

if not eth_files or not etc_files:
    print("❌ No data files found! Please run notebook 01 first to collect data.")
else:
    # Load most recent files
    eth_file = eth_files[-1]
    etc_file = etc_files[-1]
    
    print(f"Loading ETH data from: {eth_file.name}")
    print(f"Loading ETC data from: {etc_file.name}")
    
    eth_df = pl.read_parquet(eth_file)
    etc_df = pl.read_parquet(etc_file)
    
    print(f"\n✓ Loaded {len(eth_df)} ETH candles")
    print(f"✓ Loaded {len(etc_df)} ETC candles")
    print(f"\nETH date range: {eth_df['datetime'].min()} to {eth_df['datetime'].max()}")
    print(f"ETC date range: {etc_df['datetime'].min()} to {etc_df['datetime'].max()}")

In [ ]:
# Check data alignment
print(f"ETH shape: {eth_df.shape}")
print(f"ETC shape: {etc_df.shape}")
print(f"\nFirst few ETH rows:")
print(eth_df.head())

## 2. Merge and Align Data

Ensure both series have the same timestamps.

In [ ]:
# Merge on timestamp to ensure alignment
merged_df = eth_df.select(["timestamp", "datetime", pl.col("close").alias("eth_close")]).join(
    etc_df.select(["timestamp", pl.col("close").alias("etc_close")]),
    on="timestamp",
    how="inner"
)

print(f"Merged data: {merged_df.shape[0]} aligned data points")
print(f"Date range: {merged_df['datetime'].min()} to {merged_df['datetime'].max()}")
print(f"\nFirst few rows:")
print(merged_df.head())

## 3. Visual Comparison

Let's first visualize how the two assets move together.

In [ ]:
# Plot normalized prices
fig = plot_price_comparison(
    dates=merged_df["datetime"].to_list(),
    asset1_prices=merged_df["eth_close"].to_numpy(),
    asset2_prices=merged_df["etc_close"].to_numpy(),
    asset1_name="ETH",
    asset2_name="ETC",
)
fig.show()

## 4. Basic Correlation

Calculate simple Pearson correlation.

In [ ]:
correlation = merged_df.select(
    pl.corr("eth_close", "etc_close").alias("correlation")
)["correlation"][0]

print(f"Pearson Correlation: {correlation:.4f}")
print(f"\nInterpretation:")
if correlation > 0.8:
    print("  ✅ Strong positive correlation")
elif correlation > 0.5:
    print("  ✓ Moderate positive correlation")
elif correlation > 0.3:
    print("  ⚠️  Weak positive correlation")
else:
    print("  ❌ Very weak or no correlation")

print("\nNote: Correlation alone doesn't guarantee cointegration.")
print("We need to test if the spread is stationary.")

## 5. Cointegration Analysis

Perform the Engle-Granger cointegration test.

In [ ]:
# Initialize pair analysis
pair = PairAnalysis(
    asset1_prices=merged_df["eth_close"],
    asset2_prices=merged_df["etc_close"]
)

# Run cointegration test
coint_results = pair.test_cointegration()

print("="*60)
print("COINTEGRATION TEST RESULTS (Engle-Granger)")
print("="*60)
print(f"\n{coint_results['interpretation']}")
print(f"\nP-value: {coint_results['p_value']:.6f}")
print(f"Test Statistic: {coint_results['cointegration_score']:.4f}")
print(f"\nCritical Values:")
print(f"  1%:  {coint_results['critical_values']['1%']:.4f}")
print(f"  5%:  {coint_results['critical_values']['5%']:.4f}")
print(f"  10%: {coint_results['critical_values']['10%']:.4f}")
print(f"\nHedge Ratio: {coint_results['hedge_ratio']:.6f}")
print(f"Equation: ETH = {coint_results['hedge_ratio']:.4f} × ETC + {coint_results['intercept']:.2f}")

## 6. Spread Analysis

Calculate and analyze the spread.

In [ ]:
# Calculate spread
spread = pair.calculate_spread(method="ols")

# Get spread properties
spread_props = pair.analyze_spread_properties()

print("="*60)
print("SPREAD PROPERTIES")
print("="*60)
print(f"Mean:                   {spread_props['mean']:.4f}")
print(f"Std Dev:                {spread_props['std']:.4f}")
print(f"Min:                    {spread_props['min']:.4f}")
print(f"Max:                    {spread_props['max']:.4f}")
print(f"Median:                 {spread_props['median']:.4f}")
print(f"Skewness:               {spread_props['skewness']:.4f}")
print(f"Kurtosis:               {spread_props['kurtosis']:.4f}")
print(f"Autocorrelation (lag1): {spread_props['autocorr_lag1']:.4f}")

## 7. Mean Reversion Speed

Calculate the half-life of mean reversion.

In [ ]:
half_life = pair.calculate_half_life()

print("="*60)
print("MEAN REVERSION ANALYSIS")
print("="*60)
print(f"Half-life: {half_life:.2f} periods")

if half_life < 1:
    print("⚠️  Very fast mean reversion (< 1 period)")
    print("   May be too fast for practical trading")
elif half_life < 24:
    print(f"✅ Fast mean reversion (~{half_life:.1f} hours)")
    print("   Good for intraday trading")
elif half_life < 168:  # 1 week
    print(f"✓ Moderate mean reversion (~{half_life/24:.1f} days)")
    print("   Suitable for short-term trading")
elif half_life < 720:  # 1 month
    print(f"⚠️  Slow mean reversion (~{half_life/24:.1f} days)")
    print("   May require longer holding periods")
else:
    print("❌ Very slow or no mean reversion")
    print("   Not suitable for pairs trading")

## 8. Visualize Price Relationship

Scatter plot with OLS regression line.

In [ ]:
fig = plot_scatter_with_regression(
    asset2_prices=merged_df["etc_close"].to_numpy(),
    asset1_prices=merged_df["eth_close"].to_numpy(),
    hedge_ratio=coint_results["hedge_ratio"],
    intercept=coint_results["intercept"],
    asset1_name="ETH",
    asset2_name="ETC",
)
fig.show()

## 9. Visualize Spread and Z-Score

Plot the spread and z-score with trading thresholds.

In [ ]:
# Calculate z-score
zscore = pair.calculate_zscore(window=60)  # 60-period rolling window

# Plot spread analysis
fig = plot_spread_analysis(
    dates=merged_df["datetime"].to_list(),
    spread=spread,
    zscore=zscore,
    asset1_name="ETH",
    asset2_name="ETC",
    entry_threshold=2.0,
    exit_threshold=0.5,
)
fig.show()

## 10. Z-Score Distribution

Check if z-scores follow a normal distribution.

In [ ]:
fig = plot_zscore_distribution(zscore, entry_threshold=2.0)
fig.show()

# Calculate percentage of observations outside thresholds
zscore_clean = zscore[~np.isnan(zscore)]
pct_outside = (np.abs(zscore_clean) > 2.0).sum() / len(zscore_clean) * 100

print(f"\nPercentage of observations with |z-score| > 2.0: {pct_outside:.2f}%")
print(f"Number of trading opportunities: {int((np.abs(zscore_clean) > 2.0).sum())}")

if pct_outside < 2:
    print("⚠️  Very few trading opportunities")
elif pct_outside < 10:
    print("✓ Reasonable number of trading opportunities")
else:
    print("⚠️  Many extreme deviations - may indicate high volatility or instability")

## 11. Comprehensive Summary Report

Generate a complete analysis report.

In [ ]:
report = create_summary_report(pair, "ETH", "ETC")
print(report)

## 12. Save Results

Save the analysis results for future reference.

In [ ]:
# Add spread and z-score to dataframe
results_df = merged_df.with_columns([
    pl.Series("spread", spread),
    pl.Series("zscore", zscore),
])

# Save to processed data
output_path = project_root / "data" / "processed" / "eth_etc_spread_analysis.parquet"
results_df.write_parquet(output_path)
print(f"✓ Saved analysis results to: {output_path}")

# Save summary report
report_path = project_root / "data" / "results" / "eth_etc_cointegration_report.txt"
with open(report_path, "w") as f:
    f.write(report)
print(f"✓ Saved summary report to: {report_path}")

## Summary & Next Steps

In this notebook, we:
1. ✓ Loaded ETH and ETC historical data
2. ✓ Tested for cointegration using Engle-Granger test
3. ✓ Calculated hedge ratio and spread
4. ✓ Analyzed mean-reversion properties (half-life)
5. ✓ Visualized spread behavior and z-scores
6. ✓ Generated comprehensive analysis report

### Key Findings
- See the summary report above for all results
- Check if the pair is suitable for statistical arbitrage

### Next Steps
If the pair shows promise:
- Build a backtesting framework
- Implement z-score based trading strategy
- Test different entry/exit thresholds
- Calculate expected returns and risk metrics

If the pair is not suitable:
- Test other cryptocurrency pairs
- Try different asset combinations
- Consider different timeframes